In [ ]:
pip install keras --upgrade

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement keras.engine (from versions: none)
ERROR: No matching distribution found for keras.engine


In [5]:
import  pickle
import numpy as np
from numpy import genfromtxt
from collections import defaultdict
import pandas as pd
import tensorflow as tf
import keras
from keras.api.models import Model
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import csv
import re
import tabulate

# name and year extraction
pattern = r"(.+?)\s\((\d{4})\)"

def load_data():
    item_train = genfromtxt(r'C:\\Users\\Aditi\\OneDrive\\Desktop\\Data Analysis\\software\\project\\bms_mern\backend\\Recommender\\content_item_train.csv', delimiter=',')
    user_train = genfromtxt(r'C:\\Users\\Aditi\\OneDrive\\Desktop\\Data Analysis\\software\\project\\bms_mern\\backend\\Recommender\\content_user_train.csv', delimiter=',')
    y_train    = genfromtxt(r'C:\\Users\\Aditi\\OneDrive\\Desktop\\Data Analysis\\software\\project\\bms_mern\\backend\\Recommender\\content_y_train.csv', delimiter=',')
    with open(r'C:\\Users\\Aditi\\OneDrive\\Desktop\\Data Analysis\\software\\project\\bms_mern\\backend\\Recommender\\content_item_train_header.txt', newline='') as f:    #csv reader handles quoted strings better
        item_features = list(csv.reader(f))[0]
    with open(r'C:\\Users\\Aditi\\OneDrive\\Desktop\\Data Analysis\\software\\project\\bms_mern\\backend\\Recommender\\content_item_train_header.txt', newline='') as f:
        user_features = list(csv.reader(f))[0]
    item_vecs = genfromtxt(r'C:\\Users\\Aditi\\OneDrive\\Desktop\\Data Analysis\\software\\project\\bms_mern\\backend\\Recommender\\content_item_vecs.csv', delimiter=',')
       
    movie_dict = defaultdict(dict)
    count = 0
#    with open('./data/movies.csv', newline='') as csvfile:
    with open(r'C:\\Users\\Aditi\\OneDrive\\Desktop\\Data Analysis\\software\\project\\bms_mern\\backend\\Recommender\\content_movie_list.csv', newline='') as csvfile:
        reader = csv.reader(csvfile, delimiter=',', quotechar='"')
        for line in reader:
            if count == 0: 
                count +=1  #skip header
                #print(line) 
            else:
                count +=1
                movie_id = int(line[0])  
                movie_dict[movie_id]["title"] = line[1]  
                movie_dict[movie_id]["genres"] =line[2]  

    return(item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict)


def print_pred_movies(y_p, item, movie_dict, maxcount=10):
    """ print results of prediction of a new user. inputs are expected to be in
        sorted order, unscaled. """
    count = 0
    movies_listed = defaultdict(int)
    # disp = [["y_p", "movie id", "rating ave", "title", "genres"]]
    disp = {}

    for i in range(0, y_p.shape[0]):
        if count == maxcount:
            break
        count += 1
        movie_id = item[i, 0].astype(int)
        if movie_id in movies_listed:
            continue
        movies_listed[movie_id] = 1
        name = movie_dict[movie_id]['title']
        name, year = re.findall(pattern, name)[0]
        disp[name] = int(year)
    return(disp)

def gen_user_vecs(user_vec, num_items):
    """ given a user vector return:
        user predict maxtrix to match the size of item_vecs """
    user_vecs = np.tile(user_vec, (num_items, 1))
    return(user_vecs)

In [5]:
import numpy as np
import numpy.ma as ma
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split


import warnings
warnings.filterwarnings('ignore')

In [29]:
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, = load_data()

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # start of columns to use in training, user
i_s = 1  # start of columns to use in training, items

In [30]:
item_train_unscaled = item_train
user_train_unscaled = user_train
y_train_unscaled    = y_train

scalerItem = StandardScaler()
scalerItem.fit(item_train)
item_train = scalerItem.transform(item_train)

scalerUser = StandardScaler()
scalerUser.fit(user_train)
user_train = scalerUser.transform(user_train)

scalerTarget = MinMaxScaler((-1, 1))
scalerTarget.fit(y_train.reshape(-1, 1))
y_train = scalerTarget.transform(y_train.reshape(-1, 1))

item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)

In [31]:
num_user_features = 14
num_item_features = 16
num_outputs = 32
tf.random.set_seed(1)
# User network
user_NN = tf.keras.models.Sequential([
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs),
])

# Item network
item_NN = tf.keras.models.Sequential([
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs),
])

# Create the user input and process it through the user network
import tensorflow as tf

# Define the user input and process it through the user network
input_user = tf.keras.layers.Input(shape=(num_user_features,))
vu = user_NN(input_user)
vu = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vu)

# Define the item input and process it through the item network
input_item = tf.keras.layers.Input(shape=(num_item_features,))
vm = item_NN(input_item)
vm = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vm)


# Compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# Specify the inputs and output of the model
model = tf.keras.Model([input_user, input_item], output)

In [32]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)


tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], y_train, epochs=30)

Epoch 1/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 0.1303
Epoch 2/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1173
Epoch 3/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1152
Epoch 4/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1141
Epoch 5/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1130
Epoch 6/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1120
Epoch 7/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1110
Epoch 8/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.1103
Epoch 9/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.1095
Epoch 10/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1088
Epoch 11/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.1082
Epoch 12/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.1076
Epoch 13/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.1070
Epoch 14/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.1064
Epoch 15/30
1455/1455 ━━━━━━━

In [33]:
new_user_id = 5000
new_rating_ave = 0.0
new_action = 0.0
new_adventure = 5.0
new_animation = 0.0
new_childrens = 0.0
new_comedy = 0.0
new_crime = 0.0
new_documentary = 0.0
new_drama = 5.0
new_fantasy = 5.0
new_horror = 0.0
new_mystery = 5.0
new_romance = 0.0
new_scifi = 0.0
new_thriller = 0.0
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

In [34]:
def movies_predict(user_vec):
    user_vecs = gen_user_vecs(user_vec,len(item_vecs))
    suser_vecs = scalerUser.transform(user_vecs)
    sitem_vecs = scalerItem.transform(item_vecs)

# make a prediction
    y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

# unscale y prediction 
    y_pu = scalerTarget.inverse_transform(y_p)

# sort the results, highest prediction first
    sorted_index = np.argsort(-y_pu,axis=0).reshape(-1).tolist()  #negate to get largest rating first
    sorted_ypu   = y_pu[sorted_index]
    sorted_items = item_vecs[sorted_index] 

    return print_pred_movies(sorted_ypu, sorted_items, movie_dict, maxcount = 10)

In [35]:
movies_predict(user_vec)

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


{'Girl Who Leapt Through Time, The (Toki o kakeru shÃ´jo)': 2006,
 "Howl's Moving Castle (Hauru no ugoku shiro)": 2004,
 'Spirited Away (Sen to Chihiro no kamikakushi)': 2001,
 'Persepolis': 2007,
 'Lord of the Rings: The Return of the King, The': 2003,
 'City of God (Cidade de Deus)': 2002}